# Amazon 리뷰 크롤러 - 봇 감지 우회 개선 버전
# 기존 버전에서 발생한 봇 탐지 우회를 못하는 문제 해결 요청 <- 제미나이를 통해 생성된 코드 : 오류 검증 필요

## 주요 변경사항
| 항목 | 기존 | 개선 |
|------|------|------|
| 드라이버 | 기본 Chrome (봇 감지됨) | undetected-chromedriver 사용 |
| webdriver 속성 | 노출됨 | CDP 명령으로 완전히 숨김 |
| 창 닫힘 대응 | 없음 | 자동 감지 후 드라이버 재시작 |
| 봇 차단 감지 | 없음 | URL/Title 모니터링 추가 |
| 쿠키 | 미사용 | pickle로 저장 및 재사용 |
| 타이핑 방식 | send_keys 일괄 | 한 글자씩 랜덤 딜레이 입력 |
| 키워드 간 대기 | 없음 | 5~12초 랜덤 대기 추가 |

## 0. 사전 설치
```bash
pip install undetected-chromedriver
```

In [ ]:
import pandas as pd
import re
import time
import os
import random
import pickle

# ✅ 핵심 변경: undetected-chromedriver 사용 (봇 감지 우회)
import undetected_chromedriver as uc

from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchWindowException,
    WebDriverException,
    TimeoutException,
    NoSuchElementException
)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

COOKIE_FILE = "amazon_cookies.pkl"

In [ ]:
# ✅ 변경 1: 봇 감지 우회 드라이버
def start_driver():
    options = uc.ChromeOptions()
    options.add_argument('--start-maximized')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--lang=en-US')

    driver = uc.Chrome(options=options)

    # navigator.webdriver 완전히 숨기기
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': '''
            Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
            Object.defineProperty(navigator, 'languages', { get: () => ['en-US', 'en'] });
            Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3, 4, 5] });
        '''
    })
    return driver


# ✅ 변경 2: 쿠키 저장 / 불러오기 (로그인 세션 재사용)
def save_cookies(driver):
    with open(COOKIE_FILE, 'wb') as f:
        pickle.dump(driver.get_cookies(), f)
    print("[쿠키 저장 완료]")

def load_cookies(driver):
    if not os.path.exists(COOKIE_FILE):
        return False
    with open(COOKIE_FILE, 'rb') as f:
        cookies = pickle.load(f)
    driver.get('https://www.amazon.com')
    time.sleep(2)
    for cookie in cookies:
        try:
            driver.add_cookie(cookie)
        except Exception:
            pass
    driver.refresh()
    time.sleep(2)
    print("[쿠키 로드 완료 → 자동 로그인 시도]")
    return True


# ✅ 변경 3: 봇 차단 감지 함수
def is_blocked(driver):
    try:
        url = driver.current_url
        title = driver.title.lower()
        return any([
            'robot check' in title,
            'validateCaptcha' in url,
            'captcha' in url.lower(),
            'api.amazon' in url,
            ('signin' in url and 'review' not in url),
        ])
    except Exception:
        return True  # 창이 닫힌 경우도 차단으로 처리


# ✅ 변경 4: 차단 후 복구 함수
def recover_from_block(driver, target_url):
    print("\n⚠️  봇 차단 감지! CAPTCHA를 해결해주세요.")
    input("완료 후 Enter를 누르세요...")
    driver.get(target_url)
    human_delay(3, 6)
    save_cookies(driver)
    return driver

In [ ]:
def collect_reviews_from_current_page(driver, keyword="Initial"):
    reviews_data = []

    # 차단 여부 먼저 체크
    if is_blocked(driver):
        print("  → 차단 감지, 수집 중단")
        return []

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//div[starts-with(@id, 'customer_review-')]"))
        )
    except TimeoutException:
        return []

    review_blocks = driver.find_elements(By.XPATH, "//div[starts-with(@id, 'customer_review-')]")

    for block in review_blocks:
        try:
            author = block.find_element(By.XPATH, ".//div[@class='a-profile-content']/span").text
        except: author = ""

        try:
            rating_text = block.find_element(By.XPATH, ".//i[@data-hook='review-star-rating']").get_attribute("innerText")
            rating = float(rating_text.split()[0])
        except: rating = None

        try:
            review_info = block.find_element(By.XPATH, ".//span[contains(text(), 'Reviewed in')]").text
            if " on " in review_info:
                location = review_info.split("Reviewed in ")[1].split(" on ")[0].strip()
                date = review_info.split(" on ")[1].strip()
            else:
                location = ""; date = ""
        except:
            location = ""; date = ""

        try:
            opt_elem = block.find_element(By.XPATH, ".//a[@data-hook='format-strip']")
            raw_html = opt_elem.get_attribute('innerHTML')
            processed_html = re.sub(r'<i.*?</i>', ' | ', raw_html)
            clean_text = re.sub(r'<[^>]+>', '', processed_html)
            option = " ".join(clean_text.split())
        except: option = ""

        try:
            content = block.find_element(By.XPATH, ".//span[@data-hook='review-body']//span").text
        except: content = ""

        reviews_data.append({
            '검색어': keyword,
            '작성자': author,
            '별점': rating,
            '작성지': location,
            '작성일': date,
            '옵션': option,
            '리뷰 내용': content
        })

    return reviews_data

In [ ]:
def get_custom_stopwords(words_list=None):
    stop_words = set(ENGLISH_STOP_WORDS)
    if words_list:
        stop_words.update(words_list)
    return list(stop_words)


def get_combined_keywords(text_data, top_n=15, words_list=None):
    custom_stopwords = get_custom_stopwords(words_list)
    if hasattr(text_data, 'dropna'):
        text_data = text_data.dropna().tolist()

    count_vectorizer = CountVectorizer(stop_words=custom_stopwords)
    X_counts = count_vectorizer.fit_transform(text_data)
    sum_words = X_counts.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in count_vectorizer.vocabulary_.items()]
    top_words = [w[0] for w in sorted(words_freq, key=lambda x: x[1], reverse=True)[:top_n]]

    tfidf_vectorizer = TfidfVectorizer(stop_words=custom_stopwords)
    X_tfidf = tfidf_vectorizer.fit_transform(text_data)
    sum_tfidf = X_tfidf.sum(axis=0)
    words_tfidf = [(word, sum_tfidf[0, idx]) for word, idx in tfidf_vectorizer.vocabulary_.items()]
    top_tfidf = [w[0] for w in sorted(words_tfidf, key=lambda x: x[1], reverse=True)[:top_n]]

    keyword = set(top_words)
    keyword.update(top_tfidf)
    print(keyword)
    return list(keyword)


def refined_wordlist(wordlist: list):
    result = []
    wordlist = sorted(wordlist, key=lambda x: len(x))
    check_list = sorted(('s', 'es', 'ed', 'ing'), key=len, reverse=True)
    for word in wordlist:
        str_word = str(word)
        matched_suffix = None
        for suffix in check_list:
            if str_word.endswith(suffix):
                matched_suffix = suffix
                break
        if matched_suffix:
            root = str_word[:-len(matched_suffix)]
            if root not in result:
                result.append(str_word)
        else:
            result.append(str_word)
    return result


def human_delay(min_sec=2, max_sec=5):
    time.sleep(random.uniform(min_sec, max_sec))


def human_scroll(driver):
    total_height = driver.execute_script("return document.body.scrollHeight")
    current_pos = 0
    while current_pos < total_height:
        step = random.randint(300, 600)
        current_pos += step
        driver.execute_script(f"window.scrollTo(0, {current_pos});")
        time.sleep(random.uniform(0.3, 0.8))
        if current_pos > 2000:
            break
    human_delay(1, 2)

In [ ]:
# ✅ 변경 5: 창 닫힘/봇 감지 시 자동 복구 + driver 반환
def crawl_amazon_keyword(driver, keyword, target_url, max_pages=5):
    collected_data = []
    print(f"\n=== 키워드 검색 시작: '{keyword}' ===")

    # 창 닫힘 감지 및 재시작
    try:
        _ = driver.current_url
    except (NoSuchWindowException, WebDriverException):
        print("  → 창 닫힘 감지. 드라이버 재시작...")
        try:
            driver.quit()
        except Exception:
            pass
        driver = start_driver()
        load_cookies(driver)
        driver.get(target_url)
        human_delay(3, 6)

    # 봇 차단 감지 시 복구
    if is_blocked(driver):
        driver = recover_from_block(driver, target_url)

    try:
        search_box = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, '//*[@id="filterByKeywordTextBox"]'))
        )
        ActionChains(driver).move_to_element(search_box).click().perform()
        search_box.clear()

        # ✅ 한 글자씩 랜덤 딜레이로 타이핑 (봇 감지 우회)
        for char in keyword:
            search_box.send_keys(char)
            time.sleep(random.uniform(0.05, 0.15))

        human_delay(0.5, 1.5)
        search_button = driver.find_element(By.CSS_SELECTOR, '#a-autoid-1 > span > input')
        search_button.click()
        human_delay(3, 5)

    except Exception as e:
        print(f"  검색창 조작 실패 ({keyword}): {type(e).__name__}")
        return [], driver

    current_page = 1
    while current_page <= max_pages:
        print(f"  [Keyword: {keyword}] Page {current_page}...")

        if is_blocked(driver):
            driver = recover_from_block(driver, target_url)
            break

        human_scroll(driver)
        page_data = collect_reviews_from_current_page(driver, keyword)
        if not page_data:
            print("  → 리뷰 없음, 루프 종료")
            break

        collected_data.extend(page_data)

        try:
            next_button = driver.find_element(By.XPATH, "//li[@class='a-last']/a")
            if "a-disabled" in next_button.find_element(By.XPATH, "..").get_attribute("class"):
                break
            ActionChains(driver).move_to_element(next_button).perform()
            human_delay(1, 2)
            next_button.click()
            human_delay(5, 9)
            current_page += 1
        except Exception:
            break

    return collected_data, driver  # ✅ 수정된 driver 반환

In [ ]:
def split_option(df, col_name="옵션"):
    if col_name not in df.columns:
        return df

    def parse_row(row_str):
        if not isinstance(row_str, str):
            return {}
        data = {}
        for opt in row_str.split('|'):
            if ':' in opt:
                key, val = opt.split(':', 1)
                data[key.strip().lower()] = val.strip()
        return data

    split_col = df[col_name].apply(parse_row)
    option_df = pd.json_normalize(split_col)
    option_df.index = df.index
    return pd.concat([df, option_df], axis=1)

In [ ]:
# ============================
#  설정값 (여기만 수정하세요)
# ============================
PRODUCT_NAME = 'The Heaven & Earth Grocery Store - 2023'
INITIAL_URL = 'https://www.amazon.com/product-reviews/B0BPNP7YQB/ref=cm_cr_dp_d_show_all_btm?ie=UTF8&reviewerType=all_reviews'

INITIAL_MAX_PAGES = 10
KEYWORD_MAX_PAGES = 10
TOP_N_KEYWORDS = 10

MY_STOPWORDS = ['don', 'like', 've']
DEL_TITLE = PRODUCT_NAME.lower().split()
MY_STOPWORDS += DEL_TITLE

SELF_SEARCH_WORDS = ['conflict']

In [ ]:
# ============================
#  메인 실행
# ============================

# STEP 0: 드라이버 시작 + 로그인
driver = start_driver()

# 쿠키가 있으면 자동 로그인, 없으면 수동 로그인 후 저장
if not load_cookies(driver):
    driver.get(INITIAL_URL)
    print("브라우저에서 로그인 또는 CAPTCHA를 해결해주세요.")
    input("리뷰 페이지가 보이면 Enter를 누르세요...")
    save_cookies(driver)
else:
    driver.get(INITIAL_URL)
    human_delay(3, 5)
    if is_blocked(driver):
        print("차단 감지. CAPTCHA를 해결해주세요.")
        input("완료 후 Enter를 누르세요...")
        save_cookies(driver)


# STEP 1: 최초 데이터 수집
print("\nSTEP 1: 최초 리뷰 데이터 수집 중...")
initial_reviews = []
current_page = 1

while current_page <= INITIAL_MAX_PAGES:
    print(f"  Initial Page {current_page}...")

    if is_blocked(driver):
        driver = recover_from_block(driver, INITIAL_URL)

    page_data = collect_reviews_from_current_page(driver, keyword="Initial_Crawl")
    initial_reviews.extend(page_data)

    try:
        next_button = driver.find_element(By.XPATH, "//li[@class='a-last']/a")
        next_button.click()
        human_delay(3, 6)
        current_page += 1
    except Exception:
        break

df_initial = pd.DataFrame(initial_reviews)
print(f"초기 수집 완료: {len(df_initial)}개 리뷰")


# STEP 2: 키워드 추출
print("\nSTEP 2: 텍스트 분석 및 추가 키워드 추출 중...")
extracted_keywords = []

if not df_initial.empty:
    extracted_keywords = get_combined_keywords(
        text_data=df_initial['리뷰 내용'],
        top_n=TOP_N_KEYWORDS,
        words_list=MY_STOPWORDS
    )
    if extracted_keywords:
        extracted_keywords = refined_wordlist(extracted_keywords)
    print(f"추출된 키워드: {extracted_keywords}")

if SELF_SEARCH_WORDS:
    print(f'사용자 지정 단어 추가 : {SELF_SEARCH_WORDS}')
    extracted_keywords.extend(SELF_SEARCH_WORDS)

extracted_keywords = list(set(extracted_keywords))
print(f"최종 검색할 키워드 리스트: {extracted_keywords}")


# STEP 3: 키워드별 2차 크롤링
print("\nSTEP 3: 추출된 키워드로 상세 크롤링 시작...")
additional_reviews = []

for i, keyword in enumerate(extracted_keywords):
    kw_data, driver = crawl_amazon_keyword(driver, keyword, INITIAL_URL, max_pages=KEYWORD_MAX_PAGES)
    additional_reviews.extend(kw_data)

    # ✅ 키워드 사이 추가 대기 (봇 감지 방지)
    if i < len(extracted_keywords) - 1:
        wait_time = random.uniform(5, 12)
        print(f"  다음 키워드까지 {wait_time:.1f}초 대기...")
        time.sleep(wait_time)

driver.quit()


# STEP 4: 데이터 병합 및 저장
print("\nSTEP 4: 데이터 병합 및 저장 중...")
df_additional = pd.DataFrame(additional_reviews)

if not df_additional.empty:
    print(f"추가 수집된 데이터: {len(df_additional)}개")
    final_df = pd.concat([df_initial, df_additional], ignore_index=True)
else:
    print("추가 수집된 데이터가 없습니다.")
    final_df = df_initial

initial_len = len(final_df)
final_df.drop_duplicates(subset=['작성자', '리뷰 내용'], keep='first', inplace=True)
final_df = split_option(final_df, '옵션')
print(f"중복 제거: {initial_len - len(final_df)}건 삭제됨.")

os.makedirs('data', exist_ok=True)
output_filename = f"amazon_reviews_{PRODUCT_NAME}_final.csv"
final_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"\n총 {len(final_df)}개의 리뷰가 '{output_filename}'에 저장되었습니다.")